# Gold Layer 2 - ICU Clinical Monitoring

Creates an analysis-ready daily summary of clinical activity during each ICU stay.

**Grain:** One row per ICU stay per clinical day (`stay_id` + `clinical_date`).

The Gold table combines ICU stay context with selected bedside observations, laboratory activity, fluid input/output and procedures. High-volume event tables are aggregated before joining to keep the final dataset manageable for dashboards and Genie.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

silver_catalog = "cdr_silver"
silver_schema = "processed_mimic_iv"

required_tables = [
    "icustays",
    "patients",
    "admissions",
    "chartevents",
    "d_items",
    "labevents",
    "d_labitems",
    "inputevents",
    "outputevents",
    "procedureevents"
]

for table in required_tables:
    full_name = f"{silver_catalog}.{silver_schema}.{table}"
    exists = spark.catalog.tableExists(full_name)
    print(f"{table:<20} {'FOUND' if exists else 'MISSING'}")

In [0]:
icustays_df = spark.table(
    "cdr_silver.processed_mimic_iv.icustays"
)

icu_day_base_df = (
    icustays_df
    .withColumn(
        "icu_stay_complete",
        F.col("outtime").isNotNull()
    )
    .withColumn(
        "_end_date",
        F.to_date(F.coalesce(F.col("outtime"), F.col("intime")))
    )
    .withColumn(
        "clinical_date",
        F.explode(
            F.sequence(
                F.to_date("intime"),
                F.col("_end_date")
            )
        )
    )
    .withColumn(
        "icu_day_number",
        F.datediff(
            F.col("clinical_date"),
            F.to_date("intime")
        ) + 1
    )
    .select(
        "subject_id",
        "hadm_id",
        "stay_id",
        "clinical_date",
        "icu_day_number",
        F.col("first_careunit").alias("icu_first_careunit"),
        F.col("last_careunit").alias("icu_last_careunit"),
        F.col("intime").alias("icu_intime"),
        F.col("outtime").alias("icu_outtime"),
        F.col("los").alias("icu_los_days"),
        "icu_stay_complete"
    )
)

In [0]:
icu_day_base_df.select(
    F.count("*").alias("rows"),
    F.countDistinct("stay_id").alias("unique_stays"),
    F.countDistinct(
        F.struct("stay_id", "clinical_date")
    ).alias("unique_stay_days"),
    F.min("icu_day_number").alias("min_icu_day"),
    F.max("icu_day_number").alias("max_icu_day"),
    F.sum((~F.col("icu_stay_complete")).cast("int")).alias("incomplete_stay_days")
).show(truncate=False)

In [0]:
patients_df = spark.table(
    "cdr_silver.processed_mimic_iv.patients"
)

admissions_df = spark.table(
    "cdr_silver.processed_mimic_iv.admissions"
)

icu_gold_df = (
    icu_day_base_df.alias("i")

    # Admission context
    .join(
        admissions_df.alias("a"),
        F.col("i.hadm_id") == F.col("a.hadm_id"),
        "left"
    )

    # Patient context
    .join(
        patients_df.alias("p"),
        F.col("i.subject_id") == F.col("p.subject_id"),
        "left"
    )

    .select(
        # Keys / grain
        F.col("i.subject_id"),
        F.col("i.hadm_id"),
        F.col("i.stay_id"),
        F.col("i.clinical_date"),
        F.col("i.icu_day_number"),

        # Patient
        F.col("p.gender"),

        (
            F.col("p.anchor_age")
            + F.year("a.admittime")
            - F.col("p.anchor_year")
        ).alias("age_at_admission"),

        # Hospital admission
        F.col("a.admission_type"),
        F.col("a.hospital_expire_flag").alias("mortality_flag"),

        # ICU context
        F.col("i.icu_first_careunit"),
        F.col("i.icu_last_careunit"),
        F.col("i.icu_intime"),
        F.col("i.icu_outtime"),
        F.col("i.icu_los_days"),
        F.col("i.icu_stay_complete")
    )

    .withColumn(
        "age_group",
        F.when(F.col("age_at_admission") < 40, "18-39")
         .when(F.col("age_at_admission") < 60, "40-59")
         .when(F.col("age_at_admission") < 80, "60-79")
         .otherwise("80+")
    )
)

In [0]:
icu_gold_df.select(
    F.count("*").alias("rows"),
    F.countDistinct(
        F.struct("stay_id", "clinical_date")
    ).alias("unique_stay_days"),
    F.countDistinct("stay_id").alias("unique_stays"),
    F.sum(F.col("gender").isNull().cast("int")).alias("missing_gender"),
    F.sum(F.col("age_at_admission").isNull().cast("int")).alias("missing_age"),
    F.sum(F.col("admission_type").isNull().cast("int")).alias("missing_admission")
).show(truncate=False)

In [0]:
d_items_df = spark.table(
    "cdr_silver.processed_mimic_iv.d_items"
)

print("D_ITEMS SCHEMA")
d_items_df.printSchema()

print("\nCANDIDATE VITAL SIGN ITEMS")

(
    d_items_df
    .filter(
        F.lower(F.col("label")).rlike(
            "heart rate|respiratory rate|temperature|spo2|oxygen saturation|o2 saturation"
        )
    )
    .select(
        "itemid",
        "label",
        "abbreviation",
        "category",
        "unitname",
        "param_type"
    )
    .orderBy("label", "itemid")
    .show(200, truncate=False)
)

In [0]:
chartevents_df = spark.table(
    "cdr_silver.processed_mimic_iv.chartevents"
)

print("CHARTEVENTS SCHEMA")
chartevents_df.printSchema()

In [0]:
# Curated bedside vital-sign item IDs
HEART_RATE_ID = 220045
RESP_RATE_ID = 220210
SPO2_ID = 220277
TEMP_C_ID = 223762
TEMP_F_ID = 223761

vital_item_ids = [
    HEART_RATE_ID,
    RESP_RATE_ID,
    SPO2_ID,
    TEMP_C_ID,
    TEMP_F_ID
]

# Filter immediately to the small set of vitals needed for Gold
vital_events_df = (
    chartevents_df
    .filter(F.col("itemid").isin(vital_item_ids))
    .filter(F.col("valuenum").isNotNull())
    .withColumn(
        "clinical_date",
        F.to_date("charttime")
    )
    .withColumn(
        "standard_value",
        F.when(
            F.col("itemid") == TEMP_F_ID,
            (F.col("valuenum") - 32) * (5.0 / 9.0)
        ).otherwise(F.col("valuenum"))
    )
)

In [0]:
vitals_daily_df = (
    vital_events_df
    .groupBy("stay_id", "clinical_date")
    .agg(
        # Overall monitoring activity
        F.count("*").alias("vital_observation_count"),
        F.countDistinct("itemid").alias("distinct_vital_measurements"),
        F.sum(
            F.when(F.col("warning") == 1, 1).otherwise(0)
        ).alias("vital_warning_count"),

        # Heart rate
        F.round(
            F.avg(F.when(F.col("itemid") == HEART_RATE_ID, F.col("standard_value"))),
            2
        ).alias("heart_rate_avg"),
        F.round(
            F.min(F.when(F.col("itemid") == HEART_RATE_ID, F.col("standard_value"))),
            2
        ).alias("heart_rate_min"),
        F.round(
            F.max(F.when(F.col("itemid") == HEART_RATE_ID, F.col("standard_value"))),
            2
        ).alias("heart_rate_max"),

        # Respiratory rate
        F.round(
            F.avg(F.when(F.col("itemid") == RESP_RATE_ID, F.col("standard_value"))),
            2
        ).alias("resp_rate_avg"),
        F.round(
            F.min(F.when(F.col("itemid") == RESP_RATE_ID, F.col("standard_value"))),
            2
        ).alias("resp_rate_min"),
        F.round(
            F.max(F.when(F.col("itemid") == RESP_RATE_ID, F.col("standard_value"))),
            2
        ).alias("resp_rate_max"),

        # SpO2
        F.round(
            F.avg(F.when(F.col("itemid") == SPO2_ID, F.col("standard_value"))),
            2
        ).alias("spo2_avg"),
        F.round(
            F.min(F.when(F.col("itemid") == SPO2_ID, F.col("standard_value"))),
            2
        ).alias("spo2_min"),
        F.round(
            F.max(F.when(F.col("itemid") == SPO2_ID, F.col("standard_value"))),
            2
        ).alias("spo2_max"),

        # Temperature, standardised to Celsius
        F.round(
            F.avg(
                F.when(
                    F.col("itemid").isin([TEMP_C_ID, TEMP_F_ID]),
                    F.col("standard_value")
                )
            ),
            2
        ).alias("temperature_c_avg"),
        F.round(
            F.min(
                F.when(
                    F.col("itemid").isin([TEMP_C_ID, TEMP_F_ID]),
                    F.col("standard_value")
                )
            ),
            2
        ).alias("temperature_c_min"),
        F.round(
            F.max(
                F.when(
                    F.col("itemid").isin([TEMP_C_ID, TEMP_F_ID]),
                    F.col("standard_value")
                )
            ),
            2
        ).alias("temperature_c_max")
    )
)

In [0]:
vitals_daily_df.select(
    F.count("*").alias("vital_stay_days"),
    F.countDistinct("stay_id").alias("stays_with_vitals"),
    F.sum("vital_observation_count").alias("selected_vital_observations"),
    F.min("vital_observation_count").alias("min_observations_per_day"),
    F.max("vital_observation_count").alias("max_observations_per_day")
).show(truncate=False)

In [0]:
vitals_daily_table = "cdr_gold.analytics_mimic_iv._tmp_vitals_daily"

(
    vitals_daily_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(vitals_daily_table)
)

vitals_daily_df = spark.table(vitals_daily_table)

print("Daily vitals materialised successfully.")

In [0]:
vitals_daily_df.select(
    #Heart rate
    F.min("heart_rate_min").alias("hr_min"),
    F.max("heart_rate_max").alias("hr_max"),

    #Respiratory rate
    F.min("resp_rate_min").alias("rr_min"),
    F.max("resp_rate_max").alias("rr_max"),

    #SpO2
    F.min("spo2_min").alias("spo2_min"),
    F.max("spo2_max").alias("spo2_max"),

    #Temperature
    F.min("temperature_c_min").alias("temp_min_c"),
    F.max("temperature_c_max").alias("temp_max_c"),

    #Missingness by ICU-day
    F.sum(F.col("heart_rate_avg").isNull().cast("int")).alias("days_missing_hr"),
    F.sum(F.col("resp_rate_avg").isNull().cast("int")).alias("days_missing_rr"),
    F.sum(F.col("spo2_avg").isNull().cast("int")).alias("days_missing_spo2"),
    F.sum(F.col("temperature_c_avg").isNull().cast("int")).alias("days_missing_temp")
).show(truncate=False)

In [0]:
vital_events_df.select(
    F.count("*").alias("total_observations"),

    F.sum(
        (
            (F.col("itemid") == HEART_RATE_ID) &
            ~F.col("standard_value").between(20, 300)
        ).cast("int")
    ).alias("invalid_hr"),

    F.sum(
        (
            (F.col("itemid") == RESP_RATE_ID) &
            ~F.col("standard_value").between(1, 100)
        ).cast("int")
    ).alias("invalid_rr"),

    F.sum(
        (
            (F.col("itemid") == SPO2_ID) &
            ~F.col("standard_value").between(0, 100)
        ).cast("int")
    ).alias("invalid_spo2"),

    F.sum(
        (
            F.col("itemid").isin([TEMP_C_ID, TEMP_F_ID]) &
            ~F.col("standard_value").between(25, 45)
        ).cast("int")
    ).alias("invalid_temp")
).show(truncate=False)

In [0]:
vital_events_df.select(
    F.count("*").alias("total_observations"),

    F.sum(
        (
            (F.col("itemid") == HEART_RATE_ID) &
            ~F.col("standard_value").between(0, 400)
        ).cast("int")
    ).alias("invalid_hr"),

    F.sum(
        (
            (F.col("itemid") == RESP_RATE_ID) &
            ~F.col("standard_value").between(0, 150)
        ).cast("int")
    ).alias("invalid_rr"),

    F.sum(
        (
            (F.col("itemid") == SPO2_ID) &
            ~F.col("standard_value").between(0, 100)
        ).cast("int")
    ).alias("invalid_spo2"),

    F.sum(
        (
            F.col("itemid").isin([TEMP_C_ID, TEMP_F_ID]) &
            ~F.col("standard_value").between(15, 50)
        ).cast("int")
    ).alias("invalid_temp")
).show(truncate=False)

In [0]:
#Keep only technically plausible vital observations
valid_vital_events_df = vital_events_df.filter(
    (
        (F.col("itemid") == HEART_RATE_ID) &
        F.col("standard_value").between(0, 400)
    )
    |
    (
        (F.col("itemid") == RESP_RATE_ID) &
        F.col("standard_value").between(0, 150)
    )
    |
    (
        (F.col("itemid") == SPO2_ID) &
        F.col("standard_value").between(0, 100)
    )
    |
    (
        F.col("itemid").isin([TEMP_C_ID, TEMP_F_ID]) &
        F.col("standard_value").between(10, 50)
    )
)

vitals_daily_clean_df = (
    valid_vital_events_df
    .groupBy("stay_id", "clinical_date")
    .agg(
        F.count("*").alias("vital_observation_count"),
        F.countDistinct("itemid").alias("distinct_vital_measurements"),
        F.sum(
            F.when(F.col("warning") == 1, 1).otherwise(0)
        ).alias("vital_warning_count"),

        # Heart rate
        F.round(
            F.avg(F.when(F.col("itemid") == HEART_RATE_ID, F.col("standard_value"))), 2
        ).alias("heart_rate_avg"),
        F.round(
            F.min(F.when(F.col("itemid") == HEART_RATE_ID, F.col("standard_value"))), 2
        ).alias("heart_rate_min"),
        F.round(
            F.max(F.when(F.col("itemid") == HEART_RATE_ID, F.col("standard_value"))), 2
        ).alias("heart_rate_max"),

        # Respiratory rate
        F.round(
            F.avg(F.when(F.col("itemid") == RESP_RATE_ID, F.col("standard_value"))), 2
        ).alias("resp_rate_avg"),
        F.round(
            F.min(F.when(F.col("itemid") == RESP_RATE_ID, F.col("standard_value"))), 2
        ).alias("resp_rate_min"),
        F.round(
            F.max(F.when(F.col("itemid") == RESP_RATE_ID, F.col("standard_value"))), 2
        ).alias("resp_rate_max"),

        # SpO2
        F.round(
            F.avg(F.when(F.col("itemid") == SPO2_ID, F.col("standard_value"))), 2
        ).alias("spo2_avg"),
        F.round(
            F.min(F.when(F.col("itemid") == SPO2_ID, F.col("standard_value"))), 2
        ).alias("spo2_min"),
        F.round(
            F.max(F.when(F.col("itemid") == SPO2_ID, F.col("standard_value"))), 2
        ).alias("spo2_max"),

        # Temperature standardised to Celsius
        F.round(
            F.avg(
                F.when(
                    F.col("itemid").isin([TEMP_C_ID, TEMP_F_ID]),
                    F.col("standard_value")
                )
            ), 2
        ).alias("temperature_c_avg"),
        F.round(
            F.min(
                F.when(
                    F.col("itemid").isin([TEMP_C_ID, TEMP_F_ID]),
                    F.col("standard_value")
                )
            ), 2
        ).alias("temperature_c_min"),
        F.round(
            F.max(
                F.when(
                    F.col("itemid").isin([TEMP_C_ID, TEMP_F_ID]),
                    F.col("standard_value")
                )
            ), 2
        ).alias("temperature_c_max")
    )
)

# Materialise the cleaned aggregation
(
    vitals_daily_clean_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("cdr_gold.analytics_mimic_iv._tmp_vitals_daily")
)

vitals_daily_df = spark.table(
    "cdr_gold.analytics_mimic_iv._tmp_vitals_daily"
)

print("Clean daily vitals rebuilt successfully.")

In [0]:
vitals_daily_df.select(
    F.min("heart_rate_min").alias("hr_min"),
    F.max("heart_rate_max").alias("hr_max"),

    F.min("resp_rate_min").alias("rr_min"),
    F.max("resp_rate_max").alias("rr_max"),

    F.min("spo2_min").alias("spo2_min"),
    F.max("spo2_max").alias("spo2_max"),

    F.min("temperature_c_min").alias("temp_min_c"),
    F.max("temperature_c_max").alias("temp_max_c"),

    F.count("*").alias("vital_stay_days")
).show(truncate=False)

In [0]:
icu_gold_df = (
    icu_gold_df.alias("i")
    .join(
        vitals_daily_df.alias("v"),
        on=[
            F.col("i.stay_id") == F.col("v.stay_id"),
            F.col("i.clinical_date") == F.col("v.clinical_date")
        ],
        how="left"
    )
    .select(
        "i.*",

        # Monitoring activity
        F.coalesce(F.col("v.vital_observation_count"), F.lit(0))
            .alias("vital_observation_count"),

        F.coalesce(F.col("v.distinct_vital_measurements"), F.lit(0))
            .alias("distinct_vital_measurements"),

        F.coalesce(F.col("v.vital_warning_count"), F.lit(0))
            .alias("vital_warning_count"),

        # Heart rate
        F.col("v.heart_rate_avg"),
        F.col("v.heart_rate_min"),
        F.col("v.heart_rate_max"),

        # Respiratory rate
        F.col("v.resp_rate_avg"),
        F.col("v.resp_rate_min"),
        F.col("v.resp_rate_max"),

        # SpO2
        F.col("v.spo2_avg"),
        F.col("v.spo2_min"),
        F.col("v.spo2_max"),

        # Temperature
        F.col("v.temperature_c_avg"),
        F.col("v.temperature_c_min"),
        F.col("v.temperature_c_max")
    )
)

In [0]:
icu_gold_df.select(
    F.count("*").alias("rows"),
    F.countDistinct(
        F.struct("stay_id", "clinical_date")
    ).alias("unique_stay_days"),

    F.sum(
        (F.col("vital_observation_count") > 0).cast("int")
    ).alias("days_with_vitals"),

    F.sum(
        (F.col("vital_observation_count") == 0).cast("int")
    ).alias("days_without_vitals"),

    F.sum("vital_observation_count").alias("total_vital_observations")
).show(truncate=False)

In [0]:
unmatched_vital_days_df = (
    vitals_daily_df.alias("v")
    .join(
        icu_day_base_df.select(
            "stay_id",
            "clinical_date"
        ).alias("i"),
        on=[
            F.col("v.stay_id") == F.col("i.stay_id"),
            F.col("v.clinical_date") == F.col("i.clinical_date")
        ],
        how="left_anti"
    )
)

unmatched_vital_days_df.select(
    F.count("*").alias("unmatched_vital_days"),
    F.countDistinct("stay_id").alias("affected_stays"),
    F.sum("vital_observation_count").alias("unmatched_observations")
).show(truncate=False)

In [0]:
(
    unmatched_vital_days_df.alias("v")
    .join(
        icustays_df.select(
            "stay_id",
            "intime",
            "outtime"
        ).alias("i"),
        "stay_id",
        "left"
    )
    .select(
        F.sum(
            (
                F.col("v.clinical_date") < F.to_date("i.intime")
            ).cast("int")
        ).alias("before_icu_start"),

        F.sum(
            (
                F.col("i.outtime").isNotNull() &
                (F.col("v.clinical_date") > F.to_date("i.outtime"))
            ).cast("int")
        ).alias("after_icu_end"),

        F.sum(
            F.col("i.intime").isNull().cast("int")
        ).alias("missing_icu_match")
    )
    .show(truncate=False)
)

In [0]:
(
    unmatched_vital_days_df.alias("v")
    .join(
        icustays_df.select(
            "stay_id",
            "intime",
            "outtime"
        ).alias("i"),
        "stay_id",
        "left"
    )
    .select(
        F.sum(
            F.col("i.outtime").isNull().cast("int")
        ).alias("unmatched_days_in_incomplete_stays"),

        F.countDistinct(
            F.when(
                F.col("i.outtime").isNull(),
                F.col("v.stay_id")
            )
        ).alias("incomplete_stays_affected")
    )
    .show(truncate=False)
)

In [0]:
d_labitems_df = spark.table(
    "cdr_silver.processed_mimic_iv.d_labitems"
)

print("D_LABITEMS SCHEMA")
d_labitems_df.printSchema()

print("\nCANDIDATE LAB ITEMS")

(
    d_labitems_df
    .filter(
        F.lower(F.col("label")).rlike(
            "creatinine|hemoglobin|white blood cell|wbc|sodium|potassium|lactate"
        )
    )
    .select(
        "itemid",
        "label",
        "fluid",
        "category"
    )
    .orderBy("label", "itemid")
    .show(200, truncate=False)
)

In [0]:
labevents_df = spark.table(
    "cdr_silver.processed_mimic_iv.labevents"
)

print("LABEVENTS SCHEMA")
labevents_df.printSchema()

In [0]:
# Curated lab item IDs
CREATININE_IDS = [50912, 52546]
HEMOGLOBIN_IDS = [51222]
WBC_IDS = [51300, 51301]
SODIUM_IDS = [50983, 52623]
POTASSIUM_IDS = [50971, 52610]
LACTATE_IDS = [50813, 52442, 53154]

selected_lab_ids = (
    CREATININE_IDS
    + HEMOGLOBIN_IDS
    + WBC_IDS
    + SODIUM_IDS
    + POTASSIUM_IDS
    + LACTATE_IDS
)

selected_labs_df = (
    labevents_df
    .filter(F.col("itemid").isin(selected_lab_ids))
    .filter(F.col("valuenum").isNotNull())
    .select(
        "labevent_id",
        "subject_id",
        "hadm_id",
        "itemid",
        "charttime",
        "valuenum",
        "valueuom",
        "ref_range_lower",
        "ref_range_upper",
        "flag"
    )
)

(
    selected_labs_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(
        "cdr_gold.analytics_mimic_iv._tmp_selected_labs"
    )
)

selected_labs_df = spark.table(
    "cdr_gold.analytics_mimic_iv._tmp_selected_labs"
)

print("Selected lab events materialised successfully.")

In [0]:
selected_labs_df.select(
    F.count("*").alias("selected_lab_events"),
    F.countDistinct("hadm_id").alias("admissions_with_selected_labs"),
    F.countDistinct("subject_id").alias("patients_with_selected_labs"),
    F.sum(F.col("hadm_id").isNull().cast("int")).alias("missing_hadm_id"),
    F.min("charttime").alias("earliest_charttime"),
    F.max("charttime").alias("latest_charttime")
).show(truncate=False)

In [0]:
(
    selected_labs_df
    .groupBy("flag")
    .count()
    .orderBy(F.desc("count"))
    .show(truncate=False)
)

In [0]:
(
    selected_labs_df
    .groupBy(
        "itemid",
        "valueuom"
    )
    .count()
    .orderBy(
        "itemid",
        F.desc("count")
    )
    .show(100, truncate=False)
)

In [0]:
icu_stay_windows_df = (
    icustays_df
    .select(
        "subject_id",
        "hadm_id",
        "stay_id",
        "intime",
        "outtime"
    )
)

icu_labs_df = (
    selected_labs_df.alias("l")
    .join(
        F.broadcast(icu_stay_windows_df.alias("i")),
        (
            (F.col("l.subject_id") == F.col("i.subject_id"))
            &
            (
                F.col("l.hadm_id").isNull()
                |
                (F.col("l.hadm_id") == F.col("i.hadm_id"))
            )
            &
            (
                (
                    F.col("i.outtime").isNotNull()
                    &
                    F.col("l.charttime").between(
                        F.col("i.intime"),
                        F.col("i.outtime")
                    )
                )
                |
                (
                    F.col("i.outtime").isNull()
                    &
                    (F.col("l.charttime") >= F.col("i.intime"))
                    &
                    (
                        F.to_date("l.charttime")
                        == F.to_date("i.intime")
                    )
                )
            )
        ),
        "inner"
    )
    .select(
        F.col("l.labevent_id"),
        F.col("i.stay_id"),
        F.col("i.hadm_id"),
        F.to_date("l.charttime").alias("clinical_date"),
        F.col("l.itemid"),
        F.col("l.valuenum"),
        F.col("l.valueuom"),
        F.col("l.flag"),

        F.when(F.col("l.itemid").isin(CREATININE_IDS), "creatinine")
         .when(F.col("l.itemid").isin(HEMOGLOBIN_IDS), "hemoglobin")
         .when(F.col("l.itemid").isin(WBC_IDS), "wbc")
         .when(F.col("l.itemid").isin(SODIUM_IDS), "sodium")
         .when(F.col("l.itemid").isin(POTASSIUM_IDS), "potassium")
         .when(F.col("l.itemid").isin(LACTATE_IDS), "lactate")
         .alias("lab_name"),

        (F.col("l.flag") == "abnormal")
            .cast("int")
            .alias("abnormal_flag")
    )
)

In [0]:
(
    icu_labs_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(
        "cdr_gold.analytics_mimic_iv._tmp_icu_selected_labs"
    )
)

icu_labs_df = spark.table(
    "cdr_gold.analytics_mimic_iv._tmp_icu_selected_labs"
)

print("ICU lab events mapped successfully.")

In [0]:
icu_labs_df.select(
    F.count("*").alias("mapped_lab_rows"),
    F.countDistinct("labevent_id").alias("unique_labevents"),
    F.countDistinct("stay_id").alias("stays_with_labs"),
    F.sum("abnormal_flag").alias("abnormal_lab_results")
).show(truncate=False)

In [0]:
(
    icu_labs_df
    .groupBy("lab_name")
    .agg(
        F.count("*").alias("results"),
        F.min("valuenum").alias("min_value"),
        F.max("valuenum").alias("max_value"),
        F.sum("abnormal_flag").alias("abnormal_results")
    )
    .orderBy("lab_name")
    .show(truncate=False)
)

In [0]:
(
    icu_labs_df
    .filter(F.col("lab_name") == "lactate")
    .select(
        "valuenum",
        "valueuom",
        "flag",
        "stay_id",
        "clinical_date"
    )
    .orderBy(F.desc("valuenum"))
    .show(30, truncate=False)
)

In [0]:
icu_labs_df.filter(
    F.col("lab_name") == "lactate"
).select(
    F.count("*").alias("lactate_results"),

    F.sum((F.col("valuenum") > 30).cast("int"))
        .alias("above_30"),

    F.sum((F.col("valuenum") > 50).cast("int"))
        .alias("above_50"),

    F.sum((F.col("valuenum") > 100).cast("int"))
        .alias("above_100")
).show(truncate=False)

In [0]:
# Remove only clearly corrupted lactate values
clean_icu_labs_df = icu_labs_df.filter(
    (F.col("lab_name") != "lactate") |
    F.col("valuenum").between(0, 50)
)

labs_daily_df = (
    clean_icu_labs_df
    .groupBy("stay_id", "clinical_date")
    .agg(
        # Overall lab activity
        F.count("*").alias("lab_result_count"),
        F.countDistinct("lab_name").alias("distinct_lab_tests"),
        F.sum("abnormal_flag").alias("abnormal_lab_count"),

        # Selected daily lab summaries
        F.round(
            F.avg(F.when(F.col("lab_name") == "creatinine", F.col("valuenum"))),
            2
        ).alias("creatinine_avg"),

        F.round(
            F.avg(F.when(F.col("lab_name") == "hemoglobin", F.col("valuenum"))),
            2
        ).alias("hemoglobin_avg"),

        F.round(
            F.avg(F.when(F.col("lab_name") == "wbc", F.col("valuenum"))),
            2
        ).alias("wbc_avg"),

        F.round(
            F.avg(F.when(F.col("lab_name") == "sodium", F.col("valuenum"))),
            2
        ).alias("sodium_avg"),

        F.round(
            F.avg(F.when(F.col("lab_name") == "potassium", F.col("valuenum"))),
            2
        ).alias("potassium_avg"),

        F.round(
            F.avg(F.when(F.col("lab_name") == "lactate", F.col("valuenum"))),
            2
        ).alias("lactate_avg")
    )
)

(
    labs_daily_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(
        "cdr_gold.analytics_mimic_iv._tmp_labs_daily"
    )
)

labs_daily_df = spark.table(
    "cdr_gold.analytics_mimic_iv._tmp_labs_daily"
)

print("Daily ICU labs materialised successfully.")

In [0]:
labs_daily_df.select(
    F.count("*").alias("lab_stay_days"),
    F.countDistinct("stay_id").alias("stays_with_labs"),
    F.sum("lab_result_count").alias("total_lab_results"),
    F.sum("abnormal_lab_count").alias("abnormal_lab_results")
).show(truncate=False)

In [0]:
icu_gold_df = (
    icu_gold_df.alias("i")
    .join(
        labs_daily_df.alias("l"),
        on=[
            F.col("i.stay_id") == F.col("l.stay_id"),
            F.col("i.clinical_date") == F.col("l.clinical_date")
        ],
        how="left"
    )
    .select(
        "i.*",

        # Lab activity
        F.coalesce(F.col("l.lab_result_count"), F.lit(0))
            .alias("lab_result_count"),

        F.coalesce(F.col("l.distinct_lab_tests"), F.lit(0))
            .alias("distinct_lab_tests"),

        F.coalesce(F.col("l.abnormal_lab_count"), F.lit(0))
            .alias("abnormal_lab_count"),

        # Selected lab summaries
        F.col("l.creatinine_avg"),
        F.col("l.hemoglobin_avg"),
        F.col("l.wbc_avg"),
        F.col("l.sodium_avg"),
        F.col("l.potassium_avg"),
        F.col("l.lactate_avg")
    )
)

In [0]:
icu_gold_df.select(
    F.count("*").alias("rows"),

    F.countDistinct(
        F.struct("stay_id", "clinical_date")
    ).alias("unique_stay_days"),

    F.sum(
        (F.col("lab_result_count") > 0).cast("int")
    ).alias("days_with_labs"),

    F.sum(
        (F.col("lab_result_count") == 0).cast("int")
    ).alias("days_without_labs"),

    F.sum("lab_result_count").alias("total_lab_results"),

    F.sum("abnormal_lab_count").alias("abnormal_lab_results")
).show(truncate=False)

In [0]:
inputevents_df = spark.table(
    "cdr_silver.processed_mimic_iv.inputevents"
)

outputevents_df = spark.table(
    "cdr_silver.processed_mimic_iv.outputevents"
)

print("INPUTEVENTS SCHEMA")
inputevents_df.printSchema()

print("\nOUTPUTEVENTS SCHEMA")
outputevents_df.printSchema()

In [0]:
print("INPUTEVENTS AMOUNT UNITS")

(
    inputevents_df
    .groupBy("amountuom")
    .count()
    .orderBy(F.desc("count"))
    .show(50, truncate=False)
)

print("\nINPUTEVENTS STATUS")

(
    inputevents_df
    .groupBy("statusdescription")
    .count()
    .orderBy(F.desc("count"))
    .show(50, truncate=False)
)

print("\nOUTPUTEVENTS VALUE UNITS")

(
    outputevents_df
    .groupBy("valueuom")
    .count()
    .orderBy(F.desc("count"))
    .show(50, truncate=False)
)

In [0]:
volume_input_df = (
    inputevents_df
    .filter(F.col("amount").isNotNull())
    .filter(F.col("amountuom").isin(["ML", "L", "CM3", "ΜL"]))
    .withColumn(
        "amount_ml",
        F.when(F.col("amountuom") == "L", F.col("amount") * 1000)
         .when(F.col("amountuom") == "ΜL", F.col("amount") / 1000)
         .otherwise(F.col("amount"))
    )
)

volume_input_df.select(
    F.count("*").alias("volume_input_rows"),

    F.sum(F.col("starttime").isNull().cast("int"))
        .alias("missing_starttime"),

    F.sum(F.col("endtime").isNull().cast("int"))
        .alias("missing_endtime"),

    F.sum(
        (
            F.to_date("starttime") != F.to_date("endtime")
        ).cast("int")
    ).alias("crosses_calendar_day"),

    F.sum((F.col("amount_ml") < 0).cast("int"))
        .alias("negative_amount"),

    F.min("amount_ml").alias("min_amount_ml"),
    F.max("amount_ml").alias("max_amount_ml")
).show(truncate=False)

In [0]:
print("LARGEST INPUT VOLUMES")

(
    volume_input_df
    .select(
        "stay_id",
        "itemid",
        "starttime",
        "endtime",
        "amount_ml",
        "amountuom",
        "statusdescription",
        "ordercategoryname"
    )
    .orderBy(F.desc("amount_ml"))
    .show(30, truncate=False)
)

print("\nNEGATIVE INPUT VOLUMES")

(
    volume_input_df
    .filter(F.col("amount_ml") < 0)
    .select(
        "stay_id",
        "itemid",
        "starttime",
        "endtime",
        "amount_ml",
        "amountuom",
        "statusdescription",
        "ordercategoryname"
    )
    .orderBy("amount_ml")
    .show(30, truncate=False)
)


In [0]:
suspicious_input_ids = [
    225828,
    226361,
    220949,
    226364,
    226368,
    226452,
    226367,
    226377,
    225158,
    225943
]

(
    d_items_df
    .filter(F.col("itemid").isin(suspicious_input_ids))
    .select(
        "itemid",
        "label",
        "abbreviation",
        "category",
        "unitname",
        "param_type"
    )
    .orderBy("itemid")
    .show(truncate=False)
)

In [0]:
# Exclude clearly non-ICU intake records
non_icu_input_ids = [
    226361,  # Pre-Admission/Non-ICU Intake
    226364,  # OR Crystalloid Intake
    226367,  # OR FFP Intake
    226368,  # OR Packed RBC Intake
    226377   # PACU PO Intake
]

icu_volume_input_df = (
    volume_input_df
    .filter(F.col("amount_ml") >= 0)
    .filter(~F.col("itemid").isin(non_icu_input_ids))
)

icu_volume_input_df.select(
    F.count("*").alias("icu_volume_rows"),

    F.sum((F.col("amount_ml") > 10000).cast("int"))
        .alias("above_10k_ml"),

    F.sum((F.col("amount_ml") > 20000).cast("int"))
        .alias("above_20k_ml"),

    F.sum((F.col("amount_ml") > 50000).cast("int"))
        .alias("above_50k_ml"),

    F.sum((F.col("amount_ml") > 100000).cast("int"))
        .alias("above_100k_ml"),

    F.max("amount_ml").alias("max_amount_ml")
).show(truncate=False)

In [0]:
(
    icu_volume_input_df
    .select(
        "stay_id",
        "itemid",
        "starttime",
        "endtime",
        "amount_ml",
        "statusdescription",
        "ordercategoryname"
    )
    .orderBy(F.desc("amount_ml"))
    .show(20, truncate=False)
)

In [0]:
# Keep ICU-relevant volume inputs only
icu_volume_input_df = (
    volume_input_df
    .filter(F.col("amount_ml") >= 0)

    # Remove pre-admission / non-ICU summary records
    .filter(
        (F.col("ordercategoryname").isNull()) |
        (F.col("ordercategoryname") != "16-Pre Admission/Non-ICU")
    )

    # Remove known OR / PACU summary intake items
    .filter(
        ~F.col("itemid").isin([
            226364,  # OR Crystalloid Intake
            226367,  # OR FFP Intake
            226368,  # OR Packed RBC Intake
            226377   # PACU PO Intake
        ])
    )

    # Calculate event duration
    .withColumn(
        "duration_minutes",
        (
            F.col("endtime").cast("long") -
            F.col("starttime").cast("long")
        ) / 60.0
    )

    # Approximate hourly rate from recorded amount and duration
    .withColumn(
        "implied_ml_per_hour",
        F.when(
            F.col("duration_minutes") > 0,
            F.col("amount_ml") / (F.col("duration_minutes") / 60.0)
        )
    )
)

In [0]:
icu_volume_input_df.select(
    F.count("*").alias("rows"),

    F.sum(
        (F.col("duration_minutes") <= 0).cast("int")
    ).alias("invalid_duration"),

    F.sum(
        (F.col("implied_ml_per_hour") > 5000).cast("int")
    ).alias("above_5k_ml_hr"),

    F.sum(
        (F.col("implied_ml_per_hour") > 10000).cast("int")
    ).alias("above_10k_ml_hr"),

    F.sum(
        (F.col("implied_ml_per_hour") > 50000).cast("int")
    ).alias("above_50k_ml_hr"),

    F.max("implied_ml_per_hour").alias("max_implied_ml_hr")
).show(truncate=False)

In [0]:
(
    icu_volume_input_df
    .select(
        "stay_id",
        "itemid",
        "starttime",
        "endtime",
        "duration_minutes",
        "amount_ml",
        "implied_ml_per_hour",
        "statusdescription",
        "ordercategoryname"
    )
    .orderBy(F.desc("implied_ml_per_hour"))
    .show(30, truncate=False)
)

In [0]:
(
    icu_volume_input_df
    .filter(F.col("amount_ml") > 50000)
    .select(
        "stay_id",
        "itemid",
        "starttime",
        "endtime",
        "duration_minutes",
        "amount_ml",
        "statusdescription",
        "ordercategoryname"
    )
    .orderBy(F.desc("amount_ml"))
    .show(50, truncate=False)
)

In [0]:
clean_input_df = (
    icu_volume_input_df
    .filter(F.col("duration_minutes") > 0)
    .filter(F.col("amount_ml") <= 50000)
)

input_daily_df = (
    clean_input_df

    # Create one row for each calendar day touched by the event
    .withColumn(
        "clinical_date",
        F.explode(
            F.sequence(
                F.to_date("starttime"),
                F.to_date("endtime")
            )
        )
    )

    # Calendar-day boundaries
    .withColumn(
        "_day_start",
        F.col("clinical_date").cast("timestamp")
    )
    .withColumn(
        "_day_end",
        F.date_add(F.col("clinical_date"), 1).cast("timestamp")
    )

    # Portion of the event occurring inside this calendar day
    .withColumn(
        "_overlap_start",
        F.greatest(
            F.col("starttime"),
            F.col("_day_start")
        )
    )
    .withColumn(
        "_overlap_end",
        F.least(
            F.col("endtime"),
            F.col("_day_end")
        )
    )
    .withColumn(
        "_overlap_seconds",
        F.col("_overlap_end").cast("long") -
        F.col("_overlap_start").cast("long")
    )
    .filter(F.col("_overlap_seconds") > 0)

    # Allocate the recorded amount proportionally across the event duration
    .withColumn(
        "allocated_input_ml",
        F.col("amount_ml") *
        (
            F.col("_overlap_seconds") /
            (
                F.col("endtime").cast("long") -
                F.col("starttime").cast("long")
            )
        )
    )

    # Collapse to Gold grain
    .groupBy(
        "stay_id",
        "clinical_date"
    )
    .agg(
        F.round(
            F.sum("allocated_input_ml"),
            2
        ).alias("fluid_input_ml"),

        F.count("*").alias("input_event_segments")
    )
)

In [0]:
(
    input_daily_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(
        "cdr_gold.analytics_mimic_iv._tmp_input_daily"
    )
)

input_daily_df = spark.table(
    "cdr_gold.analytics_mimic_iv._tmp_input_daily"
)

print("Daily ICU fluid input materialised successfully.")

In [0]:
original_input_total = (
    clean_input_df
    .agg(
        F.sum("amount_ml").alias("original_input_ml")
    )
)

allocated_input_total = (
    input_daily_df
    .agg(
        F.sum("fluid_input_ml").alias("allocated_input_ml"),
        F.count("*").alias("input_stay_days"),
        F.countDistinct("stay_id").alias("stays_with_input")
    )
)

original_input_total.show(truncate=False)
allocated_input_total.show(truncate=False)

In [0]:
input_daily_df.select(
    F.min("fluid_input_ml").alias("min_daily_input_ml"),
    F.max("fluid_input_ml").alias("max_daily_input_ml"),
    F.sum((F.col("fluid_input_ml") < 0).cast("int"))
        .alias("negative_daily_inputs")
).show(truncate=False)

In [0]:
output_volume_df = (
    outputevents_df
    .filter(F.col("value").isNotNull())
    .withColumn(
        "clinical_date",
        F.to_date("charttime")
    )
)

output_volume_df.select(
    F.count("*").alias("output_rows"),

    F.sum(
        F.col("stay_id").isNull().cast("int")
    ).alias("missing_stay_id"),

    F.sum(
        F.col("charttime").isNull().cast("int")
    ).alias("missing_charttime"),

    F.sum(
        (F.col("value") < 0).cast("int")
    ).alias("negative_output"),

    F.min("value").alias("min_output_ml"),
    F.max("value").alias("max_output_ml")
).show(truncate=False)

In [0]:
output_labeled_df = (
    output_volume_df.alias("o")
    .join(
        d_items_df.select(
            "itemid",
            "label",
            "category"
        ).alias("d"),
        "itemid",
        "left"
    )
)

print("LARGEST OUTPUT VALUES")

(
    output_labeled_df
    .select(
        "stay_id",
        "itemid",
        "label",
        "category",
        "charttime",
        "value",
        "valueuom"
    )
    .orderBy(F.desc("value"))
    .show(30, truncate=False)
)

print("\nNEGATIVE OUTPUT VALUES")

(
    output_labeled_df
    .filter(F.col("value") < 0)
    .select(
        "stay_id",
        "itemid",
        "label",
        "category",
        "charttime",
        "value",
        "valueuom"
    )
    .orderBy("value")
    .show(30, truncate=False)
)

In [0]:
non_icu_output_ids = [
    226633,  # Pre-Admission
    226626,  # OR EBL
    226629   # PACU EBL
]

icu_output_df = (
    output_volume_df
    .filter(F.col("value") >= 0)
    .filter(~F.col("itemid").isin(non_icu_output_ids))
)

icu_output_df.select(
    F.count("*").alias("icu_output_rows"),

    F.sum((F.col("value") > 10000).cast("int"))
        .alias("above_10k_ml"),

    F.sum((F.col("value") > 20000).cast("int"))
        .alias("above_20k_ml"),

    F.sum((F.col("value") > 50000).cast("int"))
        .alias("above_50k_ml"),

    F.sum((F.col("value") > 100000).cast("int"))
        .alias("above_100k_ml"),

    F.max("value").alias("max_output_ml")
).show(truncate=False)

In [0]:
(
    icu_output_df.alias("o")
    .join(
        d_items_df.select(
            "itemid",
            "label",
            "category"
        ).alias("d"),
        "itemid",
        "left"
    )
    .select(
        "stay_id",
        "itemid",
        "label",
        "category",
        "charttime",
        "value"
    )
    .orderBy(F.desc("value"))
    .show(30, truncate=False)
)

In [0]:
(
    d_items_df
    .filter(F.lower(F.col("linksto")) == "outputevents")
    .filter(
        F.lower(F.col("label")).rlike(
            "pre-admission|^or |pacu|cath lab|irrigant"
        )
    )
    .select(
        "itemid",
        "label",
        "category",
        "unitname"
    )
    .orderBy("label", "itemid")
    .show(100, truncate=False)
)

In [0]:
excluded_output_ids = [
    # Non-ICU / procedural summaries
    226632,  # Cath Lab
    226626,  # OR EBL
    226627,  # OR Urine
    226628,  # PACU Drains
    226629,  # PACU EBL
    226630,  # PACU Gastric
    226631,  # PACU Urine
    226633,  # Pre-Admission

    # GU irrigation-related
    227487,  # GU Irrigant Type
    227488,  # GU Irrigant Volume In
    227489,  # GU Irrigant/Urine Volume Out
    226566   # Urine and GU Irrigant Out
]

clean_output_df = (
    output_volume_df
    .filter(F.col("value") >= 0)
    .filter(F.col("value") <= 50000)
    .filter(~F.col("itemid").isin(excluded_output_ids))
)

output_daily_df = (
    clean_output_df
    .groupBy(
        "stay_id",
        "clinical_date"
    )
    .agg(
        F.round(
            F.sum("value"),
            2
        ).alias("fluid_output_ml"),

        F.count("*").alias("output_event_count")
    )
)

(
    output_daily_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(
        "cdr_gold.analytics_mimic_iv._tmp_output_daily"
    )
)

output_daily_df = spark.table(
    "cdr_gold.analytics_mimic_iv._tmp_output_daily"
)

print("Daily ICU fluid output materialised successfully.")

In [0]:
output_daily_df.select(
    F.count("*").alias("output_stay_days"),
    F.countDistinct("stay_id").alias("stays_with_output"),
    F.sum("fluid_output_ml").alias("total_output_ml"),
    F.min("fluid_output_ml").alias("min_daily_output_ml"),
    F.max("fluid_output_ml").alias("max_daily_output_ml"),
    F.sum((F.col("fluid_output_ml") < 0).cast("int"))
        .alias("negative_daily_outputs")
).show(truncate=False)

In [0]:
icu_gold_df = (
    icu_gold_df.alias("g")

    .join(
        input_daily_df.alias("inp"),
        on=[
            F.col("g.stay_id") == F.col("inp.stay_id"),
            F.col("g.clinical_date") == F.col("inp.clinical_date")
        ],
        how="left"
    )

    .join(
        output_daily_df.alias("out"),
        on=[
            F.col("g.stay_id") == F.col("out.stay_id"),
            F.col("g.clinical_date") == F.col("out.clinical_date")
        ],
        how="left"
    )

    .select(
        "g.*",

        F.col("inp.fluid_input_ml").isNotNull()
            .alias("fluid_input_recorded"),

        F.col("out.fluid_output_ml").isNotNull()
            .alias("fluid_output_recorded"),

        F.coalesce(
            F.col("inp.fluid_input_ml"),
            F.lit(0.0)
        ).alias("fluid_input_ml"),

        F.coalesce(
            F.col("out.fluid_output_ml"),
            F.lit(0.0)
        ).alias("fluid_output_ml")
    )

    .withColumn(
        "net_fluid_balance_ml",
        F.round(
            F.col("fluid_input_ml") -
            F.col("fluid_output_ml"),
            2
        )
    )

    .withColumn(
        "fluid_balance_complete",
        F.col("fluid_input_recorded") &
        F.col("fluid_output_recorded")
    )
)

In [0]:
icu_gold_df.select(
    F.count("*").alias("rows"),

    F.countDistinct(
        F.struct("stay_id", "clinical_date")
    ).alias("unique_stay_days"),

    F.sum(F.col("fluid_input_recorded").cast("int"))
        .alias("days_with_input"),

    F.sum(F.col("fluid_output_recorded").cast("int"))
        .alias("days_with_output"),

    F.sum(F.col("fluid_balance_complete").cast("int"))
        .alias("days_with_both"),

    F.min("net_fluid_balance_ml").alias("min_recorded_balance_ml"),
    F.max("net_fluid_balance_ml").alias("max_recorded_balance_ml")
).show(truncate=False)

In [0]:
backbone_keys_df = icu_day_base_df.select(
    "stay_id",
    "clinical_date"
)

unmatched_input_df = (
    input_daily_df
    .join(
        backbone_keys_df,
        ["stay_id", "clinical_date"],
        "left_anti"
    )
)

unmatched_output_df = (
    output_daily_df
    .join(
        backbone_keys_df,
        ["stay_id", "clinical_date"],
        "left_anti"
    )
)

print("UNMATCHED INPUT DAYS")

unmatched_input_df.select(
    F.count("*").alias("unmatched_input_days"),
    F.countDistinct("stay_id").alias("affected_stays"),
    F.sum("fluid_input_ml").alias("unmatched_input_ml")
).show(truncate=False)

print("UNMATCHED OUTPUT DAYS")

unmatched_output_df.select(
    F.count("*").alias("unmatched_output_days"),
    F.countDistinct("stay_id").alias("affected_stays"),
    F.sum("fluid_output_ml").alias("unmatched_output_ml")
).show(truncate=False)

In [0]:
print("UNMATCHED INPUT DAY POSITION")

(
    unmatched_input_df.alias("x")
    .join(
        icustays_df.select(
            "stay_id",
            "intime",
            "outtime"
        ).alias("i"),
        "stay_id",
        "left"
    )
    .select(
        F.sum(
            (
                F.col("x.clinical_date") < F.to_date("i.intime")
            ).cast("int")
        ).alias("before_icu_start"),

        F.sum(
            (
                F.col("i.outtime").isNotNull() &
                (F.col("x.clinical_date") > F.to_date("i.outtime"))
            ).cast("int")
        ).alias("after_icu_end"),

        F.sum(
            F.col("i.outtime").isNull().cast("int")
        ).alias("incomplete_stay_days")
    )
    .show(truncate=False)
)

print("UNMATCHED OUTPUT DAY POSITION")

(
    unmatched_output_df.alias("x")
    .join(
        icustays_df.select(
            "stay_id",
            "intime",
            "outtime"
        ).alias("i"),
        "stay_id",
        "left"
    )
    .select(
        F.sum(
            (
                F.col("x.clinical_date") < F.to_date("i.intime")
            ).cast("int")
        ).alias("before_icu_start"),

        F.sum(
            (
                F.col("i.outtime").isNotNull() &
                (F.col("x.clinical_date") > F.to_date("i.outtime"))
            ).cast("int")
        ).alias("after_icu_end"),

        F.sum(
            F.col("i.outtime").isNull().cast("int")
        ).alias("incomplete_stay_days")
    )
    .show(truncate=False)
)

In [0]:
icu_gold_df = (
    icu_gold_df
    .withColumn(
        "net_fluid_balance_ml",
        F.when(
            F.col("fluid_balance_complete"),
            F.round(
                F.col("fluid_input_ml") -
                F.col("fluid_output_ml"),
                2
            )
        )
    )
)

In [0]:
icu_gold_df.select(
    F.sum(
        (
            F.col("fluid_balance_complete") &
            F.col("net_fluid_balance_ml").isNull()
        ).cast("int")
    ).alias("complete_but_null"),

    F.sum(
        (
            ~F.col("fluid_balance_complete") &
            F.col("net_fluid_balance_ml").isNotNull()
        ).cast("int")
    ).alias("incomplete_but_has_balance"),

    F.sum(
        F.col("fluid_balance_complete").cast("int")
    ).alias("days_with_valid_balance")
).show(truncate=False)

In [0]:
procedureevents_df = spark.table(
    "cdr_silver.processed_mimic_iv.procedureevents"
)

print("PROCEDUREEVENTS SCHEMA")
procedureevents_df.printSchema()

In [0]:
procedureevents_df.select(
    F.count("*").alias("rows"),
    F.countDistinct("orderid").alias("unique_orderids"),
    F.countDistinct(
        F.struct("stay_id", "orderid")
    ).alias("unique_stay_orders"),
    F.sum(F.col("stay_id").isNull().cast("int")).alias("missing_stay_id"),
    F.sum(F.col("starttime").isNull().cast("int")).alias("missing_starttime"),
    F.sum(F.col("itemid").isNull().cast("int")).alias("missing_itemid")
).show(truncate=False)

In [0]:
(
    procedureevents_df
    .groupBy("statusdescription")
    .count()
    .orderBy(F.desc("count"))
    .show(truncate=False)
)

In [0]:
procedures_daily_df = (
    procedureevents_df
    .withColumn(
        "clinical_date",
        F.to_date("starttime")
    )
    .groupBy(
        "stay_id",
        "clinical_date"
    )
    .agg(
        F.count("*").alias("procedure_count"),
        F.countDistinct("itemid").alias("distinct_procedure_count")
    )
)

(
    procedures_daily_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(
        "cdr_gold.analytics_mimic_iv._tmp_procedures_daily"
    )
)

procedures_daily_df = spark.table(
    "cdr_gold.analytics_mimic_iv._tmp_procedures_daily"
)

print("Daily ICU procedures materialised successfully.")

In [0]:
procedures_daily_df.select(
    F.count("*").alias("procedure_stay_days"),
    F.countDistinct("stay_id").alias("stays_with_procedures"),
    F.sum("procedure_count").alias("total_procedures"),
    F.max("procedure_count").alias("max_procedures_per_day")
).show(truncate=False)

In [0]:
icu_gold_df = (
    icu_gold_df.alias("g")
    .join(
        procedures_daily_df.alias("p"),
        on=[
            F.col("g.stay_id") == F.col("p.stay_id"),
            F.col("g.clinical_date") == F.col("p.clinical_date")
        ],
        how="left"
    )
    .select(
        "g.*",

        F.coalesce(
            F.col("p.procedure_count"),
            F.lit(0)
        ).alias("procedure_count"),

        F.coalesce(
            F.col("p.distinct_procedure_count"),
            F.lit(0)
        ).alias("distinct_procedure_types")
    )
)

In [0]:
icu_gold_df.select(
    F.count("*").alias("rows"),

    F.countDistinct(
        F.struct("stay_id", "clinical_date")
    ).alias("unique_stay_days"),

    F.sum(
        (F.col("procedure_count") > 0).cast("int")
    ).alias("days_with_procedures"),

    F.sum("procedure_count").alias("total_procedures"),

    F.max("procedure_count").alias("max_procedures_per_day")
).show(truncate=False)

In [0]:
unmatched_procedure_days_df = (
    procedures_daily_df.alias("p")
    .join(
        icu_day_base_df.select(
            "stay_id",
            "clinical_date"
        ).alias("i"),
        on=[
            F.col("p.stay_id") == F.col("i.stay_id"),
            F.col("p.clinical_date") == F.col("i.clinical_date")
        ],
        how="left_anti"
    )
)

unmatched_procedure_days_df.select(
    F.count("*").alias("unmatched_procedure_days"),
    F.countDistinct("stay_id").alias("affected_stays"),
    F.sum("procedure_count").alias("unmatched_procedures")
).show(truncate=False)

In [0]:
(
    unmatched_procedure_days_df.alias("p")
    .join(
        icustays_df.select(
            "stay_id",
            "intime",
            "outtime"
        ).alias("i"),
        "stay_id",
        "left"
    )
    .select(
        F.sum(
            (
                F.col("p.clinical_date") < F.to_date("i.intime")
            ).cast("int")
        ).alias("before_icu_start"),

        F.sum(
            (
                F.col("i.outtime").isNotNull() &
                (F.col("p.clinical_date") > F.to_date("i.outtime"))
            ).cast("int")
        ).alias("after_icu_end"),

        F.sum(
            F.col("i.outtime").isNull().cast("int")
        ).alias("incomplete_stay_days")
    )
    .show(truncate=False)
)

In [0]:
icu_gold_df.select(
    F.expr(
        "percentile_approx(vital_observation_count, array(0.50, 0.75, 0.90, 0.95, 0.99))"
    ).alias("vital_observation_percentiles"),

    F.expr(
        "percentile_approx(lab_result_count, array(0.50, 0.75, 0.90, 0.95, 0.99))"
    ).alias("lab_result_percentiles"),

    F.expr(
        "percentile_approx(procedure_count, array(0.50, 0.75, 0.90, 0.95, 0.99))"
    ).alias("procedure_percentiles"),

    F.expr(
        """
        percentile_approx(
            CASE
                WHEN fluid_balance_complete THEN abs(net_fluid_balance_ml)
            END,
            array(0.50, 0.75, 0.90, 0.95, 0.99)
        )
        """
    ).alias("absolute_fluid_balance_percentiles")
).show(truncate=False)

In [0]:
icu_gold_df = (
    icu_gold_df

    # Monitoring / activity flags
    .withColumn(
        "high_vital_monitoring_flag",
        F.col("vital_observation_count") >= 104
    )
    .withColumn(
        "high_lab_activity_flag",
        F.col("lab_result_count") >= 17
    )
    .withColumn(
        "high_procedure_activity_flag",
        F.col("procedure_count") >= 8
    )

    # Recorded abnormality indicators
    .withColumn(
        "vital_warning_flag",
        F.col("vital_warning_count") > 0
    )
    .withColumn(
        "abnormal_lab_flag",
        F.col("abnormal_lab_count") > 0
    )

    # Fluid balance direction
    .withColumn(
        "fluid_balance_direction",
        F.when(
            ~F.col("fluid_balance_complete"),
            "INCOMPLETE"
        )
        .when(
            F.col("net_fluid_balance_ml") > 0,
            "POSITIVE"
        )
        .when(
            F.col("net_fluid_balance_ml") < 0,
            "NEGATIVE"
        )
        .otherwise("BALANCED")
    )

    # Extreme relative to this dataset, not a clinical threshold
    .withColumn(
        "large_fluid_imbalance_flag",
        F.when(
            F.col("fluid_balance_complete"),
            F.abs(F.col("net_fluid_balance_ml")) >= 4465.29
        ).otherwise(F.lit(False))
    )
)

In [0]:
icu_gold_df.select(
    F.sum(F.col("high_vital_monitoring_flag").cast("int"))
        .alias("high_vital_days"),

    F.sum(F.col("high_lab_activity_flag").cast("int"))
        .alias("high_lab_days"),

    F.sum(F.col("high_procedure_activity_flag").cast("int"))
        .alias("high_procedure_days"),

    F.sum(F.col("vital_warning_flag").cast("int"))
        .alias("days_with_vital_warning"),

    F.sum(F.col("abnormal_lab_flag").cast("int"))
        .alias("days_with_abnormal_lab"),

    F.sum(F.col("large_fluid_imbalance_flag").cast("int"))
        .alias("large_fluid_imbalance_days")
).show(truncate=False)

In [0]:
icu_gold_df.select(
    F.countDistinct("stay_id").alias("icu_stays"),

    F.countDistinct(
        F.when(
            F.col("icu_first_careunit") != F.col("icu_last_careunit"),
            F.col("stay_id")
        )
    ).alias("stays_with_careunit_change"),

    F.sum(
        F.col("icu_first_careunit").isNull().cast("int")
    ).alias("rows_missing_first_careunit"),

    F.sum(
        F.col("icu_last_careunit").isNull().cast("int")
    ).alias("rows_missing_last_careunit")
).show(truncate=False)

In [0]:
icu_gold_df = (
    icu_gold_df
    .withColumn(
        "icu_careunit",
        F.col("icu_first_careunit")
    )
    .withColumn(
        "security_careunit",
        F.col("icu_first_careunit")
    )
)

In [0]:
(
    icu_gold_df
    .select("stay_id", "icu_careunit")
    .dropDuplicates(["stay_id"])
    .groupBy("icu_careunit")
    .count()
    .orderBy(F.desc("count"))
    .show(truncate=False)
)

In [0]:
icu_gold_df = (
    icu_gold_df
    .withColumn(
        "distinct_vital_measurements",
        F.col("heart_rate_avg").isNotNull().cast("int")
        + F.col("resp_rate_avg").isNotNull().cast("int")
        + F.col("spo2_avg").isNotNull().cast("int")
        + F.col("temperature_c_avg").isNotNull().cast("int")
    )
)

In [0]:
(
    icu_gold_df
    .groupBy("distinct_vital_measurements")
    .count()
    .orderBy("distinct_vital_measurements")
    .show()
)

In [0]:
print("FINAL GOLD 2 SCHEMA")
icu_gold_df.printSchema()

print("\nFINAL GOLD 2 QUALITY CHECK")

icu_gold_df.select(
    F.count("*").alias("rows"),

    F.countDistinct(
        F.struct("stay_id", "clinical_date")
    ).alias("unique_stay_days"),

    F.countDistinct("stay_id").alias("unique_stays"),

    F.sum(F.col("stay_id").isNull().cast("int"))
        .alias("null_stay_id"),

    F.sum(F.col("hadm_id").isNull().cast("int"))
        .alias("null_hadm_id"),

    F.sum(F.col("subject_id").isNull().cast("int"))
        .alias("null_subject_id"),

    F.sum(F.col("icu_careunit").isNull().cast("int"))
        .alias("missing_careunit"),

    F.sum((F.col("vital_observation_count") > 0).cast("int"))
        .alias("days_with_vitals"),

    F.sum((F.col("lab_result_count") > 0).cast("int"))
        .alias("days_with_labs"),

    F.sum((F.col("procedure_count") > 0).cast("int"))
        .alias("days_with_procedures"),

    F.sum(F.col("fluid_balance_complete").cast("int"))
        .alias("days_with_valid_fluid_balance")
).show(truncate=False)


In [0]:
icu_gold_df.select(
    F.sum(
        (
            F.col("distinct_vital_measurements") < 0
        ).cast("int")
    ).alias("invalid_vital_measurement_count_low"),

    F.sum(
        (
            F.col("distinct_vital_measurements") > 4
        ).cast("int")
    ).alias("invalid_vital_measurement_count_high"),

    F.sum(
        (
            (F.col("vital_observation_count") == 0) &
            (
                F.col("heart_rate_avg").isNotNull() |
                F.col("resp_rate_avg").isNotNull() |
                F.col("spo2_avg").isNotNull() |
                F.col("temperature_c_avg").isNotNull()
            )
        ).cast("int")
    ).alias("vitals_present_but_count_zero"),

    F.sum(
        (
            F.col("fluid_balance_complete") &
            F.col("net_fluid_balance_ml").isNull()
        ).cast("int")
    ).alias("complete_balance_but_null"),

    F.sum(
        (
            ~F.col("fluid_balance_complete") &
            F.col("net_fluid_balance_ml").isNotNull()
        ).cast("int")
    ).alias("incomplete_balance_has_value"),

    F.sum(
        (F.col("fluid_input_ml") < 0).cast("int")
    ).alias("negative_input"),

    F.sum(
        (F.col("fluid_output_ml") < 0).cast("int")
    ).alias("negative_output"),

    F.sum(
        (F.col("procedure_count") < 0).cast("int")
    ).alias("negative_procedure_count")
).show(truncate=False)

In [0]:
final_icu_gold_df = (
    icu_gold_df
    .select(
        # Keys / grain
        "subject_id",
        "hadm_id",
        "stay_id",
        "clinical_date",
        "icu_day_number",

        # Patient / admission context
        "gender",
        "age_at_admission",
        "age_group",
        "admission_type",
        F.col("mortality_flag").alias("admission_mortality_flag"),

        # ICU context
        "icu_careunit",
        "security_careunit",
        "icu_intime",
        "icu_outtime",
        "icu_los_days",
        "icu_stay_complete",

        # Vital monitoring
        "vital_observation_count",
        "distinct_vital_measurements",
        "vital_warning_count",
        "vital_warning_flag",

        "heart_rate_avg",
        "heart_rate_min",
        "heart_rate_max",

        "resp_rate_avg",
        "resp_rate_min",
        "resp_rate_max",

        "spo2_avg",
        "spo2_min",
        "spo2_max",

        "temperature_c_avg",
        "temperature_c_min",
        "temperature_c_max",

        # Laboratory activity
        "lab_result_count",
        "distinct_lab_tests",
        "abnormal_lab_count",
        "abnormal_lab_flag",

        "creatinine_avg",
        "hemoglobin_avg",
        "wbc_avg",
        "sodium_avg",
        "potassium_avg",
        "lactate_avg",

        # Fluid balance
        "fluid_input_recorded",
        "fluid_output_recorded",
        "fluid_input_ml",
        "fluid_output_ml",
        "net_fluid_balance_ml",
        "fluid_balance_complete",
        "fluid_balance_direction",

        # Procedures
        "procedure_count",
        "distinct_procedure_types",

        # Analytical activity flags
        "high_vital_monitoring_flag",
        "high_lab_activity_flag",
        "high_procedure_activity_flag",

        F.coalesce(
            F.col("large_fluid_imbalance_flag"),
            F.lit(False)
        ).alias("large_fluid_imbalance_flag")
    )
    .withColumn(
        "_gold_timestamp",
        F.current_timestamp()
    )
)

target_table = "cdr_gold.analytics_mimic_iv.icu_clinical_monitoring"

(
    final_icu_gold_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(target_table)
)

print(f"Gold table created: {target_table}")

In [0]:
published_icu_gold_df = spark.table(
    "cdr_gold.analytics_mimic_iv.icu_clinical_monitoring"
)

published_icu_gold_df.select(
    F.count("*").alias("rows"),

    F.countDistinct(
        F.struct("stay_id", "clinical_date")
    ).alias("unique_stay_days"),

    F.countDistinct("stay_id").alias("unique_stays"),

    F.sum(
        (F.col("vital_observation_count") > 0).cast("int")
    ).alias("days_with_vitals"),

    F.sum(
        (F.col("lab_result_count") > 0).cast("int")
    ).alias("days_with_labs"),

    F.sum(
        (F.col("procedure_count") > 0).cast("int")
    ).alias("days_with_procedures"),

    F.sum(
        F.col("fluid_balance_complete").cast("int")
    ).alias("days_with_valid_fluid_balance")
).show(truncate=False)

In [0]:
table_name = "cdr_gold.analytics_mimic_iv.icu_clinical_monitoring"

# Table-level metadata
spark.sql(f"""
COMMENT ON TABLE {table_name} IS
'Daily ICU clinical monitoring Gold dataset for MIMIC-IV. One row represents one ICU stay on one clinical date, identified by stay_id and clinical_date. Summarises patient and ICU context, selected bedside vital signs, selected laboratory results, fluid balance, procedure activity and analytical activity indicators. High-activity and large-fluid-imbalance flags are dataset-relative analytical indicators and are not clinical risk thresholds.'
""")

column_comments = {
    # Keys / grain
    "subject_id":
        "Deidentified patient identifier.",

    "hadm_id":
        "Hospital admission identifier associated with the ICU stay.",

    "stay_id":
        "Unique ICU stay identifier.",

    "clinical_date":
        "Calendar date represented by this ICU monitoring row. Together with stay_id, defines the grain of the table.",

    "icu_day_number":
        "Sequential ICU day number beginning at 1 on the ICU admission date.",

    # Patient / admission
    "gender":
        "Recorded patient gender from the MIMIC-IV patient table.",

    "age_at_admission":
        "Derived patient age at the time of hospital admission.",

    "age_group":
        "Derived age band based on age at hospital admission.",

    "admission_type":
        "Hospital admission type associated with the ICU stay.",

    "admission_mortality_flag":
        "Indicates whether the patient died during the overall hospital admission. This is an admission-level outcome repeated across ICU days.",

    # ICU
    "icu_careunit":
        "Care unit associated with the ICU stay. Used as the main care-unit dimension for ICU analysis.",

    "security_careunit":
        "Care-unit field intended for care-unit-based row-level security.",

    "icu_intime":
        "Timestamp when the ICU stay began.",

    "icu_outtime":
        "Timestamp when the ICU stay ended. May be null for incomplete ICU stays.",

    "icu_los_days":
        "Recorded ICU length of stay in days.",

    "icu_stay_complete":
        "Indicates whether an ICU outtime is available for the stay.",

    # Vital monitoring
    "vital_observation_count":
        "Number of retained bedside vital-sign observations recorded for the ICU stay-day.",

    "distinct_vital_measurements":
        "Number of selected vital-sign concepts available for the day, from 0 to 4: heart rate, respiratory rate, SpO2 and temperature.",

    "vital_warning_count":
        "Number of selected vital observations carrying the source warning indicator.",

    "vital_warning_flag":
        "True when at least one selected vital observation has a source warning indicator.",

    "heart_rate_avg":
        "Daily average heart rate in beats per minute after broad plausibility filtering.",

    "heart_rate_min":
        "Daily minimum heart rate in beats per minute after broad plausibility filtering.",

    "heart_rate_max":
        "Daily maximum heart rate in beats per minute after broad plausibility filtering.",

    "resp_rate_avg":
        "Daily average respiratory rate in breaths per minute after broad plausibility filtering.",

    "resp_rate_min":
        "Daily minimum respiratory rate in breaths per minute after broad plausibility filtering.",

    "resp_rate_max":
        "Daily maximum respiratory rate in breaths per minute after broad plausibility filtering.",

    "spo2_avg":
        "Daily average pulse oximetry oxygen saturation percentage after broad plausibility filtering.",

    "spo2_min":
        "Daily minimum pulse oximetry oxygen saturation percentage after broad plausibility filtering.",

    "spo2_max":
        "Daily maximum pulse oximetry oxygen saturation percentage after broad plausibility filtering.",

    "temperature_c_avg":
        "Daily average temperature in degrees Celsius. Fahrenheit observations are converted to Celsius.",

    "temperature_c_min":
        "Daily minimum temperature in degrees Celsius.",

    "temperature_c_max":
        "Daily maximum temperature in degrees Celsius.",

    # Labs
    "lab_result_count":
        "Number of retained selected laboratory results for the ICU stay-day.",

    "distinct_lab_tests":
        "Number of selected laboratory concepts represented on the ICU stay-day.",

    "abnormal_lab_count":
        "Number of selected laboratory results explicitly flagged as abnormal in the source data.",

    "abnormal_lab_flag":
        "True when at least one selected laboratory result is explicitly flagged as abnormal.",

    "creatinine_avg":
        "Daily average blood creatinine result in mg/dL.",

    "hemoglobin_avg":
        "Daily average blood hemoglobin result in g/dL.",

    "wbc_avg":
        "Daily average blood white blood cell count in K/uL.",

    "sodium_avg":
        "Daily average blood sodium result in mEq/L.",

    "potassium_avg":
        "Daily average blood potassium result in mEq/L.",

    "lactate_avg":
        "Daily average blood lactate result in mmol/L after removal of clearly corrupted extreme values.",

    # Fluid balance
    "fluid_input_recorded":
        "Indicates whether retained ICU fluid input data is available for the stay-day.",

    "fluid_output_recorded":
        "Indicates whether retained ICU fluid output data is available for the stay-day.",

    "fluid_input_ml":
        "Total retained ICU fluid input allocated to the clinical day in millilitres.",

    "fluid_output_ml":
        "Total retained ICU fluid output recorded on the clinical day in millilitres.",

    "net_fluid_balance_ml":
        "Daily fluid input minus fluid output in millilitres. Populated only when both input and output are recorded.",

    "fluid_balance_complete":
        "True when both retained fluid input and retained fluid output are available for the stay-day.",

    "fluid_balance_direction":
        "Direction of daily fluid balance: POSITIVE, NEGATIVE, BALANCED or INCOMPLETE.",

    # Procedures
    "procedure_count":
        "Number of ICU procedure orders beginning on the clinical day.",

    "distinct_procedure_types":
        "Number of distinct procedure item IDs beginning on the clinical day.",

    # Analytical flags
    "high_vital_monitoring_flag":
        "Dataset-relative analytical flag for ICU days with at least 104 retained vital observations, based on approximately the 95th percentile. Not a clinical risk threshold.",

    "high_lab_activity_flag":
        "Dataset-relative analytical flag for ICU days with at least 17 selected lab results, based on approximately the 95th percentile. Not a clinical risk threshold.",

    "high_procedure_activity_flag":
        "Dataset-relative analytical flag for ICU days with at least 8 procedures, based on approximately the 95th percentile. Not a clinical risk threshold.",

    "large_fluid_imbalance_flag":
        "Dataset-relative analytical flag where absolute complete daily fluid balance is at least 4465.29 mL, approximately the 95th percentile. Not a clinical risk threshold.",

    "_gold_timestamp":
        "Timestamp when the Gold table row was generated."
}

for column, description in column_comments.items():
    safe_description = description.replace("'", "''")

    spark.sql(f"""
        ALTER TABLE {table_name}
        ALTER COLUMN {column}
        COMMENT '{safe_description}'
    """)

print("Gold Layer 2 metadata added successfully.")